In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
from tensorflow.keras.optimizers import AdamW

In [ ]:
pip install tensorflow

In [2]:
# =========================================================
# Load and prepare data
# =========================================================
df = pd.read_csv("data/process_data/news/daily_topic_sentiment_price.csv")

# Select input and output columns
features = ['volume', 'T_minus_1_price', 'topic_0', 'topic_1', 'topic_2', 'topic_3', 
            'topic_4', 'topic_5', 'topic_6', 'topic_7', 
            'min_sentiment', 'max_sentiment', 'avg_sentiment']
target = 'change_vs_T_minus_1_price'

In [7]:
X = df[features].fillna(0)
y = df[target].fillna(0)

# Split into train-validation-test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, shuffle=False)

In [8]:
# Standardize input features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [12]:
# =========================================================
# Define learning rate schedule (warm-up ratio)
# =========================================================
initial_lr = 1e-3
warmup_ratio = 0.1
total_epochs = 20

def lr_schedule(epoch):
    if epoch < warmup_ratio * total_epochs:
        return initial_lr * (epoch + 1) / (warmup_ratio * total_epochs)
    else:
        return initial_lr * np.exp(-0.2 * (epoch - warmup_ratio * total_epochs))

In [15]:
# =========================================================
# Define NN model (4 layers)
# =========================================================
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='linear')  # Regression output
])

# =========================================================
# Compile model (AdamW optimizer)
# =========================================================
optimizer = AdamW(learning_rate=initial_lr, weight_decay=1e-4)

model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=['mae']
)

# =========================================================
# Callbacks: EarlyStopping + LR Scheduler
# =========================================================
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,   # early stopping = 4
    restore_best_weights=True
)

lr_scheduler = LearningRateScheduler(lr_schedule)

# =========================================================
# Train model
# =========================================================
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,             
    batch_size=16,         
    # callbacks=[early_stop, lr_scheduler],
    callbacks=[lr_scheduler],
    verbose=1
)

Epoch 1/100


/Users/dauvudangkhoi/Document/Text_Analytics/crawl/lib/python3.13/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 208.3647 - mae: 11.0994 - val_loss: 106.7832 - val_mae: 7.2206 - learning_rate: 5.0000e-04
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 208.3356 - mae: 11.1161 - val_loss: 106.4994 - val_mae: 7.1769 - learning_rate: 0.0010
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 207.8405 - mae: 11.0472 - val_loss: 106.4999 - val_mae: 7.1451 - learning_rate: 0.0010
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 205.5405 - mae: 10.9989 - val_loss: 106.5134 - val_mae: 7.1344 - learning_rate: 8.1873e-04
Epoch 5/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 204.3293 - mae: 10.9825 - val_loss: 106.3879 - val_mae: 7.1410 - learning_rate: 6.7032e-04
Epoch 6/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 203.9984 - mae: 10.9413 - val_loss: 106.4112 - val_mae: 7.1545 - learning_rate: 5.4881e-04
Epoch 7/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 203.3501 - mae: 10.9382 - val_loss: 106.4031 - val_mae: 7.1758 - learning_

In [16]:
# =========================================================
# Evaluate
# =========================================================
loss, mae = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f" Test MSE: {loss:.6f}, Test MAE: {mae:.6f}")

✅ Test MSE: 65.469864, Test MAE: 6.422310
